# Whisper-small — финальная модель на полных данных

**Цель:** обучить продакшн-модель на всех 2776 записях (train+val+test объединены).  
**Гиперпараметры:** взяты из `04_pretrained_finetuned/exp_whisper_small_finetune` без изменений.  
**Порог:** 0.19 — оптимальный по val F1-bad из предыдущего эксперимента.  
**Выход:** `best_ckpt.pt` и `threshold.json` — готовы для деплоя в `models/whisper_small_finetuned/`.

| Параметр | Значение |
|---|---|
| Базовая модель | openai/whisper-small |
| Размороженных слоёв | 4 (верхних) |
| lr encoder / head | 5e-6 / 1e-4 |
| Warmup | 2 эпохи |
| Early stopping patience | 10 эпох (по val 10%) |
| Порог классификации | 0.19 (фиксирован из exp_whisper_small_finetune) |

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import WhisperModel, WhisperProcessor

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils, train_utils
from shared.evaluate import find_optimal_threshold

train_utils.set_seed(config.RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(device)}")

In [ ]:
# Загружаем ВСЕ данные без выделения тестовой выборки
paths_all, labels_all, df_all = data_utils.load_dataset_csv()
aux_cols = [c for c in df_all.columns
            if c not in {"filename", "dir", "label", "path"} and df_all[c].dtype != object]
letters_all = df_all[aux_cols].values.astype(np.float32)

print(f"Всего записей: {len(paths_all)}")
print(f"  good={int((labels_all == 0).sum())}  bad={int((labels_all == 1).sum())}")

# 10% — внутренний val только для early stopping; в продакшн-оценку не входит
idx = np.arange(len(paths_all))
idx_tr, idx_val = train_test_split(
    idx, test_size=0.10, stratify=labels_all, random_state=config.RANDOM_STATE
)
paths_train,  paths_val  = paths_all[idx_tr],  paths_all[idx_val]
labels_train, labels_val = labels_all[idx_tr], labels_all[idx_val]
letters_train, letters_val = letters_all[idx_tr], letters_all[idx_val]

print(f"\nTrain: {len(paths_train)}  Val(early-stop): {len(paths_val)}")

In [ ]:
MODEL_ID = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(MODEL_ID)


class WhisperDataset(Dataset):
    def __init__(self, paths, labels, letters, augment=False):
        self.paths = paths
        self.labels = labels
        self.letters = letters
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        y, sr = data_utils.load_audio(self.paths[i], sr=16000)
        feats = processor.feature_extractor(
            y, sampling_rate=16000, return_tensors="np"
        ).input_features[0]
        if self.augment:
            feats = data_utils.augment_mel_spectrogram(feats)
        return (
            torch.from_numpy(feats).float(),
            torch.from_numpy(self.letters[i]).float(),
            int(self.labels[i]),
        )


BATCH_SIZE = 8
train_loader = DataLoader(
    WhisperDataset(paths_train, labels_train, letters_train, augment=True),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    WhisperDataset(paths_val, labels_val, letters_val, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=(device.type == "cuda"),
)
print(f"Батчей train: {len(train_loader)}  val: {len(val_loader)}")

In [ ]:
N_UNFREEZE = 4
EMBED_DIM  = 768
DROPOUT    = 0.3
N_EPOCHS   = 30


class WhisperClassifier(nn.Module):
    def __init__(self, whisper_model, n_letters=0, dropout=DROPOUT, n_unfreeze=N_UNFREEZE):
        super().__init__()
        self.encoder = whisper_model.encoder
        self.n_letters = n_letters

        for p in self.encoder.parameters():
            p.requires_grad = False

        n_layers = len(self.encoder.layers)
        for layer in self.encoder.layers[n_layers - n_unfreeze:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in self.encoder.layer_norm.parameters():
            p.requires_grad = True

        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(EMBED_DIM + n_letters, 2)

        n_frozen = sum(not p.requires_grad for p in self.encoder.parameters())
        n_train  = sum(p.requires_grad for p in self.parameters())
        print(f"Encoder заморожено: {n_frozen}  |  Обучаемых параметров: {n_train:,}")

    def forward(self, input_features, letters=None):
        hidden = self.encoder(input_features).last_hidden_state
        pooled = self.dropout(hidden.mean(dim=1))
        if self.n_letters > 0 and letters is not None:
            pooled = torch.cat([pooled, letters], dim=1)
        return self.head(pooled)


whisper_model = WhisperModel.from_pretrained(MODEL_ID)
n_letters = letters_train.shape[1]
model = WhisperClassifier(whisper_model, n_letters=n_letters).to(device)

n_params_total     = sum(p.numel() for p in model.parameters())
n_params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Всего: {n_params_total:,}  |  Обучаемых: {n_params_trainable:,}")

In [ ]:
TARGET_LR_ENCODER = 5e-6
N_WARMUP_EPOCHS   = 2

weights   = compute_class_weight("balanced", classes=np.unique(labels_train), y=labels_train)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device))

encoder_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params    = list(model.head.parameters())
optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": TARGET_LR_ENCODER, "weight_decay": 1e-3},
    {"params": head_params,    "lr": 1e-4,              "weight_decay": 1e-4},
])
scheduler      = train_utils.get_lr_scheduler(optimizer)
early_stopping = train_utils.EarlyStopping(patience=config.EARLY_STOPPING_PATIENCE)
best_ckpt_path = exp_dir / "best_ckpt.pt"

warmup_steps = N_WARMUP_EPOCHS * len(train_loader)
best_f1      = -1.0
global_step  = 0
train_losses, val_f1s = [], []


def eval_loader(loader):
    model.eval()
    logits_list, true_list = [], []
    with torch.no_grad():
        for feats, letters, y in loader:
            logits = model(feats.to(device), letters.to(device))
            logits_list.append(logits.cpu().numpy())
            true_list.extend(y.tolist())
    logits = np.concatenate(logits_list)
    proba  = torch.softmax(torch.from_numpy(logits), dim=1).numpy()[:, config.CLASS_BAD]
    return proba, np.array(true_list)


t0 = time.perf_counter()

for epoch in range(N_EPOCHS):
    model.train()
    losses = []
    for feats, letters, y in train_loader:
        if global_step < warmup_steps:
            optimizer.param_groups[0]["lr"] = TARGET_LR_ENCODER * (global_step + 1) / warmup_steps

        feats, letters, y = feats.to(device), letters.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(feats, letters), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.DEFAULT_GRAD_CLIP)
        optimizer.step()
        global_step += 1
        losses.append(loss.item())

    train_loss = np.mean(losses)
    train_losses.append(train_loss)

    val_proba, val_true = eval_loader(val_loader)
    opt_thr = find_optimal_threshold(val_true, val_proba)
    from sklearn.metrics import f1_score as _f1
    val_f1 = _f1(val_true, (val_proba >= opt_thr).astype(int),
                 pos_label=config.CLASS_BAD, average="binary")
    val_f1s.append(val_f1)

    if val_f1 > best_f1:
        best_f1 = val_f1
        train_utils.save_best_checkpoint(model, best_ckpt_path)

    if global_step > warmup_steps:
        scheduler.step(val_f1)

    print(f"Epoch {epoch+1:2d}/{N_EPOCHS}  loss={train_loss:.4f}  "
          f"val_f1_bad={val_f1:.4f}  thr={opt_thr:.2f}  "
          f"lr={optimizer.param_groups[0]['lr']:.2e}")

    if early_stopping.step(val_f1):
        print(f"Early stopping на эпохе {epoch+1}")
        break

train_time_sec = time.perf_counter() - t0
train_utils.load_best_checkpoint(model, best_ckpt_path, device)
print(f"\nОбучение: {train_time_sec:.1f} с | best val_f1_bad={best_f1:.4f}")

In [ ]:
# Пересчитываем оптимальный порог на val-выборке текущей модели
val_proba_final, val_true_final = eval_loader(val_loader)
production_threshold = find_optimal_threshold(val_true_final, val_proba_final)
print(f"Оптимальный порог (val 10%): {production_threshold:.2f}")

threshold_path = exp_dir / "threshold.json"
with open(threshold_path, "w") as f:
    json.dump({"threshold": float(production_threshold)}, f)

print(f"Порог сохранён: {threshold_path}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(train_losses, label="train_loss")
axes[0].set_title("Train loss")
axes[0].legend()
axes[1].plot(val_f1s, label="val_f1_bad")
axes[1].axhline(best_f1, color="r", linestyle="--", label=f"best={best_f1:.3f}")
axes[1].set_title("Val F1-bad (early-stop 10%)")
axes[1].legend()
plt.suptitle("Whisper-small — финальная модель на полных данных")
plt.tight_layout()
fig.savefig(exp_dir / "training_curves.png", dpi=120)
plt.show()
print(f"best_ckpt.pt: {best_ckpt_path}")
print(f"threshold.json: {threshold_path}")
print(f"\nДля деплоя скопировать оба файла в models/whisper_small_finetuned/")